In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
import tensorflow as tf

print(tf.config.list_physical_devices("GPU"))

[]


In [4]:
import sentencepiece as spm

TOKENIZER_PATH = (
    "/content/drive/MyDrive/GPT_Text_Summarization/"
    "tokenizer/gpt_tokenizer.model"
)

tokenizer = spm.SentencePieceProcessor()
tokenizer.load(TOKENIZER_PATH)

VOCAB_SIZE = tokenizer.get_piece_size()

print("Vocabulary size:", VOCAB_SIZE)

Vocabulary size: 10000


In [5]:
from datasets import load_dataset

dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

train_data = dataset["train"]
val_data = dataset["validation"]
test_data = dataset["test"]

print("Training samples:", len(train_data))
print("Validation samples:", len(val_data))
print("Testing samples:", len(test_data))

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00002-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  259MB            

... (5 outputs left out)

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Training samples: 287113
Validation samples: 13368
Testing samples: 11490


In [6]:
sample = train_data[0]

print("Article:", sample["article"][:300])
print("\nSummary:", sample["highlights"])

article_ids = tokenizer.encode(sample["article"])
summary_ids = tokenizer.encode(sample["highlights"])

print("\nArticle tokens:", len(article_ids))
print("Summary tokens:", len(summary_ids))

Article: LONDON, England (Reuters) -- Harry Potter star Daniel Radcliffe gains access to a reported £20 million ($41.1 million) fortune as he turns 18 on Monday, but he insists the money won't cast a spell on him. Daniel Radcliffe as Harry Potter in "Harry Potter and the Order of the Phoenix" To the disappoi

Summary: Harry Potter star Daniel Radcliffe gets £20M fortune as he turns 18 Monday .
Young actor says he has no plans to fritter his cash away .
Radcliffe's earnings from first five Potter films have been held in trust fund .

Article tokens: 653
Summary tokens: 55


In [7]:
import os
import sys

MODEL_DIR = "/content/drive/MyDrive/GPT_Text_Summarization/model"

print("Python path:", MODEL_DIR in sys.path)
print("Directory contents:", os.listdir(MODEL_DIR))

for filename in os.listdir(MODEL_DIR):
    print(repr(filename))

Python path: False
Directory contents: ['gpt_model_development.py', '__pycache__']
'gpt_model_development.py'
'__pycache__'


In [8]:
import importlib.util

file_path = os.path.join(MODEL_DIR, "gpt_model_development.py")

print("File exists:", os.path.isfile(file_path))
print("Module found:", importlib.util.find_spec("gpt_model_development"))
print("File size:", os.path.getsize(file_path) if os.path.isfile(file_path) else "Not found")

File exists: True
Module found: None
File size: 6706


In [9]:
!pip install tensorflow

In [10]:
import importlib.util
import os

file_path = os.path.join(MODEL_DIR, "gpt_model_development.py")

spec = importlib.util.spec_from_file_location(
    "gpt_model_development",
    file_path
)

gpt_module = importlib.util.module_from_spec(spec)
sys.modules["gpt_model_development"] = gpt_module

spec.loader.exec_module(gpt_module)

print("Module loaded successfully!")

Test 1
Input shape: (1, 5, 256)
Output shape: (1, 5, 256)

Test 2
Input shape: (2, 10, 256)
Output shape: (2, 10, 256)
Input shape: (1, 5, 256)
Output shape: (1, 5, 256)
Input shape: (1, 5, 256)
Output shape: (1, 5, 256)
Input shape: (2, 10)
Output shape: (2, 10, 10000)
Total parameters: 8420624
(self, input_ids, training=False, attention_mask=None)
Input shape: (2, 10)
Attention mask shape: (2, 10)
Output shape: (2, 10, 10000)
Module loaded successfully!


In [11]:
GPTDecoder = gpt_module.GPTDecoder

print(GPTDecoder)

<class 'gpt_model_development.GPTDecoder'>


In [12]:
MAX_LENGTH = 1024

In [13]:
VOCAB_SIZE = tokenizer.get_piece_size()

D_MODEL = 256
NUM_HEADS = 8
DFF = 1024
NUM_LAYERS = 4
DROPOUT_RATE = 0.1

In [14]:
model = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    num_heads=NUM_HEADS,
    max_length=MAX_LENGTH,
    d_model=D_MODEL,
    dff=DFF,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
)

In [15]:
PAD_ID = tokenizer.pad_id()
UNK_ID = tokenizer.unk_id()
BOS_ID = tokenizer.bos_id()
EOS_ID = tokenizer.eos_id()

print("PAD:", PAD_ID)
print("UNK:", UNK_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

PAD: 0
UNK: 1
BOS: 2
EOS: 3


In [16]:
MAX_LENGTH = 1024

PROMPT_IDS = tokenizer.encode("Summarize:", out_type=int)

def preprocess_example(example):

    article_ids = tokenizer.encode(
        example["article"], out_type=int
    )

    summary_ids = tokenizer.encode(
        example["highlights"], out_type=int
    )

    # Reserve space for BOS, prompt, separator EOS and summary EOS
    available_article_length = (
        MAX_LENGTH - len(PROMPT_IDS) - len(summary_ids) - 3
    )

    # Preserve summary; truncate article if necessary
    article_ids = article_ids[:max(0, available_article_length)]

    # Complete sequence
    sequence = (
        [BOS_ID]
        + PROMPT_IDS
        + article_ids
        + [EOS_ID]
        + summary_ids
        + [EOS_ID]
    )

    # Mask prompt and article from loss calculation
    prompt_length = 1 + len(PROMPT_IDS) + len(article_ids) + 1

    # Construct labels aligned with next-token prediction
    labels = (
    [-100] * prompt_length
    + summary_ids
    + [EOS_ID]
)

# Shift input sequence and labels
    input_ids = sequence[:-1]
    labels = labels[1:]

    return {
    "input_ids": input_ids,
    "labels": labels
}

In [17]:
processed = preprocess_example(train_data[0])

input_ids = processed["input_ids"]
labels = processed["labels"]

print("Input length:", len(input_ids))
print("Label length:", len(labels))

valid_labels = [x for x in labels if x != -100]

print("Training tokens:", len(valid_labels))
print("Last training label:", valid_labels[-1])
print("EOS ID:", EOS_ID)

assert len(input_ids) == len(labels)
assert valid_labels[-1] == EOS_ID

print("Alignment verification passed!")

Input length: 714
Label length: 714
Training tokens: 56
Last training label: 3
EOS ID: 3
Alignment verification passed!


In [18]:
import numpy as np
import tensorflow as tf


PAD_LABEL_ID = -100

def create_batch(examples):

    batch_input_ids = []
    batch_labels = []
    batch_attention_masks = []

    # Find longest sequence in this batch
    max_batch_length = max(
        len(example["input_ids"]) for example in examples
    )

    for example in examples:

        input_ids = example["input_ids"]
        labels = example["labels"]

        padding_length = max_batch_length - len(input_ids)

        # Pad input IDs
        padded_input_ids = (
            input_ids + [PAD_ID] * padding_length
        )

        # Pad labels
        padded_labels = (
            labels + [PAD_LABEL_ID] * padding_length
        )

        # Attention mask: 1 = real token, 0 = padding
        attention_mask = (
            [1] * len(input_ids) + [0] * padding_length
        )

        batch_input_ids.append(padded_input_ids)
        batch_labels.append(padded_labels)
        batch_attention_masks.append(attention_mask)

    return {
        "input_ids": tf.constant(batch_input_ids, dtype=tf.int32),
        "labels": tf.constant(batch_labels, dtype=tf.int32),
        "attention_mask": tf.constant(
            batch_attention_masks, dtype=tf.int32
        )
    }

In [19]:
sample_examples = [
    preprocess_example(train_data[0]),
    preprocess_example(train_data[1])
]

batch = create_batch(sample_examples)

print("Input shape:", batch["input_ids"].shape)
print("Labels shape:", batch["labels"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

Input shape: (2, 1023)
Labels shape: (2, 1023)
Attention mask shape: (2, 1023)


In [20]:
print("First example attention mask:")
print(batch["attention_mask"][0].numpy())

print("\nSecond example attention mask:")
print(batch["attention_mask"][1].numpy())

print("\nFirst example labels:")
print(batch["labels"][0].numpy())

First example attention mask:
[1 1 1 ... 0 0 0]

Second example attention mask:
[1 1 1 ... 1 1 1]

First example labels:
[-100 -100 -100 ... -100 -100 -100]


In [21]:
logits = model(
    batch["input_ids"],
    training=False,
    attention_mask=batch["attention_mask"]
)

print("GPT output shape:", logits.shape)

GPT output shape: (2, 1023, 10000)


In [22]:
BATCH_SIZE = 2

def data_generator(data, limit=None):

    count = len(data) if limit is None else min(len(data), limit)

    for i in range(count):
        example = preprocess_example(data[i])

        yield {
            "input_ids": example["input_ids"],
            "labels": example["labels"]
        }

In [23]:
train_dataset = tf.data.Dataset.from_generator(
    lambda: data_generator(train_data, limit=100),
    output_signature={
        "input_ids": tf.TensorSpec(shape=(None,), dtype=tf.int32),
        "labels": tf.TensorSpec(shape=(None,), dtype=tf.int32)
    }
)

In [24]:
def pad_batch(batch):
    return {
        "input_ids": batch["input_ids"].to_tensor(
            default_value=PAD_ID
        ),
        "labels": batch["labels"].to_tensor(
            default_value=-100
        )
    }

train_dataset = train_dataset.padded_batch(
    BATCH_SIZE,
    padded_shapes={
        "input_ids": [None],
        "labels": [None]
    },
    padding_values={
        "input_ids": tf.constant(PAD_ID, dtype=tf.int32),
        "labels": tf.constant(-100, dtype=tf.int32)
    }
)

In [25]:
def add_attention_mask(batch):
    attention_mask = tf.cast(
        tf.not_equal(batch["input_ids"], PAD_ID),
        tf.int32
    )

    return {
        "input_ids": batch["input_ids"],
        "labels": batch["labels"],
        "attention_mask": attention_mask
    }

train_dataset = train_dataset.map(add_attention_mask)

In [26]:
for batch in train_dataset.take(1):

    print("Input IDs:", batch["input_ids"].shape)
    print("Labels:", batch["labels"].shape)
    print("Attention mask:", batch["attention_mask"].shape)

    logits = model(
        batch["input_ids"],
        training=False,
        attention_mask=batch["attention_mask"]
    )

    print("Logits:", logits.shape)

Input IDs: (2, 1023)
Labels: (2, 1023)
Attention mask: (2, 1023)
Logits: (2, 1023, 10000)


In [27]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy(
    from_logits=True,
    reduction="none"
)

def masked_loss(labels, logits):

    # Identify valid labels
    mask = tf.not_equal(labels, -100)

    # Replace ignored labels with a valid token ID
    safe_labels = tf.where(mask, labels, 0)

    # Calculate token-level loss
    loss = loss_fn(safe_labels, logits)

    # Ignore masked positions
    mask = tf.cast(mask, loss.dtype)
    loss *= mask

    # Average over valid tokens only
    return tf.reduce_sum(loss) / tf.reduce_sum(mask)

In [28]:
for batch in train_dataset.take(1):

    logits = model(
        batch["input_ids"],
        training=False,
        attention_mask=batch["attention_mask"]
    )

    loss = masked_loss(batch["labels"], logits)

    print("Masked training loss:", loss.numpy())

Masked training loss: 9.24216


In [29]:
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=0.01
)

In [30]:
BATCH_SIZE = 2
SEQUENCE_LENGTH = 1023

def create_tf_dataset(data, limit=None, shuffle=False):

    dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(data, limit=limit),
        output_signature={
            "input_ids": tf.TensorSpec(
                shape=(None,), dtype=tf.int32
            ),
            "labels": tf.TensorSpec(
                shape=(None,), dtype=tf.int32
            )
        }
    )

    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=2000,
            reshuffle_each_iteration=True
        )

    dataset = dataset.padded_batch(
        BATCH_SIZE,
        padded_shapes={
            "input_ids": [SEQUENCE_LENGTH],
            "labels": [SEQUENCE_LENGTH]
        },
        padding_values={
            "input_ids": tf.constant(PAD_ID, dtype=tf.int32),
            "labels": tf.constant(-100, dtype=tf.int32)
        },
        drop_remainder=False
    )

    def add_attention_mask(batch):

        attention_mask = tf.cast(
            tf.not_equal(batch["input_ids"], PAD_ID),
            tf.int32
        )

        return {
            "input_ids": batch["input_ids"],
            "labels": batch["labels"],
            "attention_mask": attention_mask
        }

    return dataset.map(
        add_attention_mask,
        num_parallel_calls=tf.data.AUTOTUNE
    ).prefetch(tf.data.AUTOTUNE)

In [31]:
train_dataset = create_tf_dataset(
    train_data,
    limit=100,
    shuffle=True
)

val_dataset = create_tf_dataset(
    val_data,
    limit=20
)

In [32]:
@tf.function(reduce_retracing=True)
def train_step(batch):

    with tf.GradientTape() as tape:

        logits = model(
            batch["input_ids"],
            training=True,
            attention_mask=batch["attention_mask"]
        )

        loss = masked_loss(batch["labels"], logits)

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss

In [33]:
!pip install -q tqdm

In [34]:
@tf.function(reduce_retracing=True)
def validation_step(batch):

    logits = model(
        batch["input_ids"],
        training=False,
        attention_mask=batch["attention_mask"]
    )

    loss = masked_loss(batch["labels"], logits)

    return loss

In [35]:
from tqdm.auto import tqdm
import time

EPOCHS = 3

history = {
    "train_loss": [],
    "val_loss": []
}

for epoch in range(EPOCHS):

    print(f"\n{'='*55}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*55}")

    start_time = time.time()

    # ---------------- TRAINING ----------------

    train_losses = []

    progress_bar = tqdm(
        train_dataset,
        desc="Training",
        unit="batch"
    )

    for batch in progress_bar:

        loss = train_step(batch)

        train_losses.append(float(loss.numpy()))

        progress_bar.set_postfix(
            loss=f"{loss.numpy():.4f}"
        )

    epoch_train_loss = sum(train_losses) / len(train_losses)

    # ---------------- VALIDATION ----------------

    val_losses = []

    progress_bar = tqdm(
        val_dataset,
        desc="Validation",
        unit="batch"
    )

    for batch in progress_bar:

        loss = validation_step(batch)

        val_losses.append(float(loss.numpy()))

        progress_bar.set_postfix(
            loss=f"{loss.numpy():.4f}"
        )

    epoch_val_loss = sum(val_losses) / len(val_losses)

    # ---------------- RESULTS ----------------

    elapsed_time = time.time() - start_time

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)

    print(f"\nEpoch {epoch + 1} Results")
    print(f"Training Loss:   {epoch_train_loss:.4f}")
    print(f"Validation Loss: {epoch_val_loss:.4f}")
    print(f"Time Taken:      {elapsed_time:.2f} seconds")


Epoch 1/3


Training: 0batch [00:00, ?batch/s]

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(history["train_loss"], label="Training Loss")
plt.plot(history["val_loss"], label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("GPT Training and Validation Loss")

plt.legend()
plt.grid(True)
plt.show()

In [36]:
CHECKPOINT_DIR = (
    "/content/drive/MyDrive/GPT_Text_Summarization/checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.save_weights(
    os.path.join(CHECKPOINT_DIR, "gpt_epoch3.weights.h5")
)

In [ ]:
train_dataset = create_tf_dataset(
    train_data,
    limit=None,
    shuffle=True
)

val_dataset = create_tf_dataset(
    val_data,
    limit=None,
    shuffle=False
)

In [ ]:
import time
import tensorflow as tf

# Warm-up: allow TensorFlow to initialize
for batch in train_dataset.take(2):
    _ = train_step(batch)

# Actual benchmark
start_time = time.time()

for step, batch in enumerate(train_dataset.take(20)):
    loss = train_step(batch)

    if step % 5 == 0:
        print(
            f"Step {step + 1}/20 | "
            f"Loss: {loss.numpy():.4f}"
        )

elapsed = time.time() - start_time

print("\nTraining benchmark")
print("Total time:", round(elapsed, 2), "seconds")
print("Average per batch:", round(elapsed / 20, 3), "seconds")
print(
    "Estimated batches per hour:",
    round(3600 / (elapsed / 20))
)

In [39]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [ ]:
import tensorflow as tf
import time
import gc

def benchmark_batch_size(batch_size):

    global BATCH_SIZE, train_dataset

    BATCH_SIZE = batch_size

    train_dataset = create_tf_dataset(
        train_data,
        limit=256,
        shuffle=False
    )

    try:
        # Warm-up
        for batch in train_dataset.take(2):
            _ = train_step(batch)

        start = time.time()
        total_examples = 0

        for batch in train_dataset.take(10):

            loss = train_step(batch)
            total_examples += int(batch["input_ids"].shape[0])

        elapsed = time.time() - start

        return {
            "batch_size": batch_size,
            "seconds": round(elapsed, 2),
            "examples_per_second": round(
                total_examples / elapsed, 2
            ),
            "status": "Success"
        }

    except tf.errors.ResourceExhaustedError:
        return {
            "batch_size": batch_size,
            "status": "Out of GPU memory"
        }

    finally:
        gc.collect()

In [ ]:
results = []

for bs in [8, 16, 32, 64]:

    print(f"\nTesting batch size: {bs}")

    result = benchmark_batch_size(bs)
    results.append(result)

    print(result)

    if result["status"] != "Success":
        break

In [ ]:
SEED = 42

train_subset = train_data.shuffle(
    seed=SEED
).select(range(50000))

val_subset = val_data.shuffle(
    seed=SEED
).select(range(2000))

test_subset = test_data.shuffle(
    seed=SEED
).select(range(1000))

print("Training:", len(train_subset))
print("Validation:", len(val_subset))
print("Testing:", len(test_subset))

In [37]:
import json

subset_info = {
    "seed": SEED,
    "train_ids": [train_subset[i]["id"] for i in range(len(train_subset))],
    "validation_ids": [val_subset[i]["id"] for i in range(len(val_subset))],
    "test_ids": [test_subset[i]["id"] for i in range(len(test_subset))]
}

with open(
    "/content/drive/MyDrive/GPT_Text_Summarization/subset_ids.json",
    "w"
) as f:
    json.dump(subset_info, f)

NameError: name 'SEED' is not defined

In [ ]:
BATCH_SIZE = 32
MAX_LENGTH = 1024
EPOCHS = 5

LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

In [ ]:
model = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    num_heads=NUM_HEADS,
    max_length=MAX_LENGTH,
    d_model=D_MODEL,
    dff=DFF,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
)

In [ ]:
for batch in train_dataset.take(1):

    print("Input IDs:", batch["input_ids"].shape)
    print("Labels:", batch["labels"].shape)
    print("Attention mask:", batch["attention_mask"].shape)

    logits = model(
        batch["input_ids"],
        training=False,
        attention_mask=batch["attention_mask"]
    )

    print("Logits:", logits.shape)

In [ ]:
@tf.function(reduce_retracing=True)
def train_step(batch):

    with tf.GradientTape() as tape:

        logits = model(
            batch["input_ids"],
            training=True,
            attention_mask=batch["attention_mask"]
        )

        labels = batch["labels"]
        mask = tf.not_equal(labels, -100)
        safe_labels = tf.where(mask, labels, 0)

        token_losses = loss_fn(safe_labels, logits)

        mask_float = tf.cast(mask, token_losses.dtype)

        loss_sum = tf.reduce_sum(token_losses * mask_float)
        token_count = tf.reduce_sum(mask_float)

        loss = loss_sum / token_count

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss_sum, token_count

In [ ]:
@tf.function(reduce_retracing=True)
def validation_step(batch):

    logits = model(
        batch["input_ids"],
        training=False,
        attention_mask=batch["attention_mask"]
    )

    labels = batch["labels"]
    mask = tf.not_equal(labels, -100)
    safe_labels = tf.where(mask, labels, 0)

    token_losses = loss_fn(safe_labels, logits)
    mask_float = tf.cast(mask, token_losses.dtype)

    loss_sum = tf.reduce_sum(token_losses * mask_float)
    token_count = tf.reduce_sum(mask_float)

    return loss_sum, token_count

In [ ]:
import os

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/GPT_Text_Summarization/checkpoints"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR,
    "best_gpt.weights.h5"
)

In [ ]:
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [ ]:
dummy_input = tf.zeros((1, 10), dtype=tf.int32)
dummy_mask = tf.ones((1, 10), dtype=tf.int32)

_ = model(
    dummy_input,
    training=False,
    attention_mask=dummy_mask
)

In [ ]:
model.load_weights(BEST_MODEL_PATH)

print("Epoch 1 checkpoint loaded successfully!")

In [ ]:
def preprocess_example(example):
    article_ids = tokenizer.encode(example["article"], out_type=int)
    summary_ids = tokenizer.encode(example["highlights"], out_type=int)

    # Reserve space for BOS, prompt, separator EOS and final EOS
    max_summary_length = MAX_LENGTH - len(PROMPT_IDS) - 3
    summary_ids = summary_ids[:max_summary_length]

    available_article_length = (
        MAX_LENGTH - len(PROMPT_IDS) - len(summary_ids) - 3
    )

    article_ids = article_ids[:available_article_length]

    sequence = (
        [BOS_ID] + PROMPT_IDS +
        article_ids + [EOS_ID] +
        summary_ids + [EOS_ID]
    )

    prompt_length = 1 + len(PROMPT_IDS) + len(article_ids) + 1

    labels = (
        [-100] * prompt_length +
        summary_ids + [EOS_ID]
    )

    return {
        "input_ids": sequence[:-1],
        "labels": labels[1:]
    }

In [ ]:
batch = next(iter(train_dataset))

loss_sum, token_count = train_step(batch)

print("Loss:", float(loss_sum / token_count))
print("Valid tokens:", int(token_count))

In [ ]:
train_dataset = create_tf_dataset(
    train_subset, shuffle=True
)

val_dataset = create_tf_dataset(
    val_subset, shuffle=False
)

In [ ]:
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=0.01
)
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 0.01

optimizer = tf.keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

optimizer.build(model.trainable_variables)

print("Optimizer initialized!")
optimizer.build(model.trainable_variables)

In [ ]:
!nvidia-smi

In [ ]:

from tqdm.auto import tqdm
import time
import os

EPOCHS = 5

# Results from the completed first epoch
best_val_loss = 5.4006

patience = 2
wait = 0

history = {
    "train_loss": [],
    "val_loss": []
}

# Resume from Epoch 2
for epoch in range(1, EPOCHS):

    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")

    start_time = time.time()

    # Training
    total_train_loss = 0.0
    total_train_tokens = 0.0

    train_bar = tqdm(
        train_dataset,
        total=1563,
        desc="Training",
        unit="batch"
    )

    for batch in train_bar:

        loss_sum, token_count = train_step(batch)

        total_train_loss += float(loss_sum.numpy())
        total_train_tokens += float(token_count.numpy())

        train_bar.set_postfix(
            loss=f"{total_train_loss / total_train_tokens:.4f}"
        )

    epoch_train_loss = (
        total_train_loss / total_train_tokens
    )

    # Validation
    total_val_loss = 0.0
    total_val_tokens = 0.0

    val_bar = tqdm(
        val_dataset,
        total=63,
        desc="Validation",
        unit="batch"
    )

    for batch in val_bar:

        loss_sum, token_count = validation_step(batch)

        total_val_loss += float(loss_sum.numpy())
        total_val_tokens += float(token_count.numpy())

        val_bar.set_postfix(
            loss=f"{total_val_loss / total_val_tokens:.4f}"
        )

    epoch_val_loss = total_val_loss / total_val_tokens

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)

    # Save best model
    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        wait = 0

        model.save_weights(BEST_MODEL_PATH)

        print("Best model checkpoint saved!")

    else:
        wait += 1

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch + 1} Results")
    print(f"Training loss:   {epoch_train_loss:.4f}")
    print(f"Validation loss: {epoch_val_loss:.4f}")
    print(f"Best val loss:   {best_val_loss:.4f}")
    print(f"Time taken:      {elapsed / 60:.2f} minutes")

    if wait >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining completed!")


In [ ]:
EPOCH5_PATH = os.path.join(
    CHECKPOINT_DIR,
    "gpt_epoch5.weights.h5"
)

model.save_weights(EPOCH5_PATH)

print("Epoch 5 weights saved!")

In [ ]:
model = GPTDecoder(
    vocab_size=VOCAB_SIZE,
    num_heads=NUM_HEADS,
    max_length=MAX_LENGTH,
    d_model=D_MODEL,
    dff=DFF,
    num_layers=NUM_LAYERS,
    dropout_rate=DROPOUT_RATE
)

_ = model(
    tf.zeros((1, 10), dtype=tf.int32),
    training=False,
    attention_mask=tf.ones((1, 10), dtype=tf.int32)
)

model.load_weights(EPOCH5_PATH)

In [ ]:
optimizer = tf.keras.optimizers.AdamW(
    learning_rate=3e-4,
    weight_decay=0.01
)

optimizer.build(model.trainable_variables)

In [ ]:
EPOCHS = 10

# Best validation loss achieved during previous training
best_val_loss = 4.6280

# Early stopping
patience = 2
wait = 0

history = {
    "train_loss": [],
    "val_loss": []
}

In [ ]:

from tqdm.auto import tqdm
import time
import os

# Continue training from Epoch 5
EPOCHS = 10

# Best validation loss achieved in previous training
best_val_loss = 4.6280

patience = 2
wait = 0

history = {
    "train_loss": [],
    "val_loss": []
}

# Resume from Epoch 6
for epoch in range(5, EPOCHS):

    print(f"\n{'='*60}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*60}")

    start_time = time.time()

    # -------------------------
    # Training
    # -------------------------

    total_train_loss = 0.0
    total_train_tokens = 0.0

    train_bar = tqdm(
        train_dataset,
        total=1563,
        desc="Training",
        unit="batch"
    )

    for batch in train_bar:

        loss_sum, token_count = train_step(batch)

        total_train_loss += float(loss_sum.numpy())
        total_train_tokens += float(token_count.numpy())

        train_bar.set_postfix(
            loss=f"{total_train_loss / total_train_tokens:.4f}"
        )

    epoch_train_loss = (
        total_train_loss / total_train_tokens
    )

    # -------------------------
    # Validation
    # -------------------------

    total_val_loss = 0.0
    total_val_tokens = 0.0

    val_bar = tqdm(
        val_dataset,
        total=63,
        desc="Validation",
        unit="batch"
    )

    for batch in val_bar:

        loss_sum, token_count = validation_step(batch)

        total_val_loss += float(loss_sum.numpy())
        total_val_tokens += float(token_count.numpy())

        val_bar.set_postfix(
            loss=f"{total_val_loss / total_val_tokens:.4f}"
        )

    epoch_val_loss = (
        total_val_loss / total_val_tokens
    )

    # -------------------------
    # Record history
    # -------------------------

    history["train_loss"].append(epoch_train_loss)
    history["val_loss"].append(epoch_val_loss)

    # -------------------------
    # Save best model
    # -------------------------

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss
        wait = 0

        model.save_weights(BEST_MODEL_PATH)

        print("Best model checkpoint saved!")

    else:
        wait += 1

    # -------------------------
    # Save latest model
    # -------------------------

    latest_path = os.path.join(
        CHECKPOINT_DIR,
        f"gpt_epoch{epoch + 1}.weights.h5"
    )

    model.save_weights(latest_path)

    # -------------------------
    # Epoch results
    # -------------------------

    elapsed = time.time() - start_time

    print(f"\nEpoch {epoch + 1} Results")
    print(f"Training loss:   {epoch_train_loss:.4f}")
    print(f"Validation loss: {epoch_val_loss:.4f}")
    print(f"Best val loss:   {best_val_loss:.4f}")
    print(f"Time taken:      {elapsed / 60:.2f} minutes")

    print(f"Checkpoint saved: {latest_path}")

    # -------------------------
    # Early stopping
    # -------------------------

    if wait >= patience:
        print("Early stopping triggered.")
        break

print("\nTraining completed!")
